# Notebook 3 — Predizione e analisi del corpus
**Tesi**: Stance detection sui commenti YouTube come predittore del referendum 2026

Questo notebook alimenta il **Capitolo 4** della tesi (accuratezza predittiva, RQ1) e include le analisi richieste dal TODO §1.5: cross-tabulazione Governo×Referendum, topic modeling, short vs long video, top canali e top testate.

| Sezione | Cosa fa | Riferimento tesi |
|---------|---------|------------------|
| **§1** Setup e dataset | Carica `risultati_inferenza_completa` referendum + governo + stampa | Cap 3, §3.3 |
| **§2** Aggregato grezzo vs deduplicato | Calcola Sì% sui due aggregati, scarto vs reale 46,1% | Cap 4, §4.2 (H1) |
| **§3** Anatomia del bias | Distribuzione commenti/autore, posizione dei "rumorosi" | Cap 4, §4.3 |
| **§4** Top canali e top testate | Volume e polarizzazione (TODO §1.5) | Cap 4, §4.4 |
| **§5** Short vs Long video | Distribuzione stance per `Is_Short` (TODO §1.5) | Cap 4, §4.4 |
| **§6** Sankey Governo×Referendum | Chi è pro-governo come ha votato? (TODO §1.5) | Cap 4, §4.5 |
| **§7** Topic modeling LDA | Di cosa parlano i commenti? (TODO §1.5) | Cap 4, §4.6 |

**Input attesi (Kaggle):**
- `risultati_inferenza_completa (2).csv` — referendum con `PREDIZIONE_STANCE`
- `inferenza_definitiva.csv` — governo con `PREDIZIONE_STANCE`
- `dataset_articoli_referendum.csv` — stampa con `STANCE` etichettata

**Output principale:** verifica numerica di **H1: errore deduplicato < 2 pp** (atteso: 0,8 pp).

## §1 — Setup e caricamento dataset di inferenza

Carico i tre dataset prodotti dal notebook 02 (referendum + governo) e il dataset stampa
preparato esternamente. Le funzioni di lettura sono robuste a separatore (`,` vs `;`) e
tollerano righe corrotte (`on_bad_lines='skip'`).

**Convenzione etichette:** `POS = pro-Sì / pro-governo`, `NEG = pro-No / contro-governo`, `NEU = neutro`.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Path Kaggle (modificali se cambia la struttura input)
PATH_REFERENDUM = "/kaggle/input/datasets/francescochiarolanza/commenti-youtube-referendum-giustizia/risultati_inferenza_completa (2).csv"
PATH_GOVERNO    = "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/inferenza_definitiva.csv"
PATH_STAMPA     = "/kaggle/input/datasets/francescochiarolanza/titoli-giornali-referendum-giustizia/dataset_articoli_referendum.csv"

# Risultato ufficiale referendum 22-23 marzo 2026
SI_REAL = 46.1
NO_REAL = 53.9


def carica_csv(path, nome):
    """Lettura CSV robusta: prova prima la virgola, poi il punto e virgola."""
    try:
        df = pd.read_csv(path, dtype=str, encoding="utf-8-sig",
                         engine="python", on_bad_lines="skip")
        if len(df.columns) <= 3:
            df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                             engine="python", on_bad_lines="skip")
    except Exception as e:
        print(f"❌ Errore caricamento {nome}: {e}")
        return pd.DataFrame()
    df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
    print(f"✅ {nome}: {len(df):,} righe, {len(df.columns)} colonne")
    return df


df_ref = carica_csv(PATH_REFERENDUM, "Referendum")
df_gov = carica_csv(PATH_GOVERNO,    "Governo")
df_sta = carica_csv(PATH_STAMPA,     "Stampa")

# Normalizzo la colonna STANCE per ciascun dataset
for df, col in [(df_ref, "PREDIZIONE_STANCE"), (df_gov, "PREDIZIONE_STANCE")]:
    if col in df.columns:
        df["STANCE"] = df[col].astype(str).str.strip().str.upper()

# Stampa: cerca dinamicamente la colonna stance
stance_col_st = next((c for c in df_sta.columns if "stance" in c.lower()), None)
if stance_col_st:
    df_sta["STANCE"] = df_sta[stance_col_st].astype(str).str.strip().str.upper()

print(f"\nColonne referendum: {df_ref.columns.tolist()[:8]}...")
print(f"Colonne stampa:    {df_sta.columns.tolist()[:8]}...")

## §2 — Aggregato grezzo vs deduplicato (verifica H1)

Questo è il **risultato centrale del Capitolo 4**: confronto i due aggregati e li metto a
confronto con il risultato reale del referendum.

- **Aggregato grezzo:** conta i commenti. Ogni utente pesa quanto commenta.
  Misura il *volume del dibattito* (engagement).
- **Aggregato deduplicato ("una testa, un voto"):** un autore = un voto sull'intero periodo (`drop_duplicates(subset=['Autore'], keep='last')`).
  Misura la *distribuzione delle opinioni* nella popolazione di utenti.

**Ipotesi H1 (Cap 1):** l'aggregato deduplicato si discosta di < 2 pp dal risultato reale (46,1% di Sì).
**Risultato atteso:** errore 0,8 pp (deduplicato), errore 2,9 pp (grezzo).

> **Nota sulla finestra:** la tesi (§3.4.5) restringe il calcolo finale ai 14 giorni precedenti il voto. Qui calcolo entrambe le varianti — su tutti i dati e su finestra 14gg — per documentare che la conclusione tiene in entrambe le specifiche.

In [ ]:
# Filtra ai soli commenti con stance interpretabile
df_r = df_ref[df_ref["STANCE"].isin(["POS", "NEG"])].copy()
df_r["Data"] = pd.to_datetime(df_r["Data_Commento"], errors="coerce", dayfirst=True, utc=True)
df_r = df_r.dropna(subset=["Data"])
df_r["Data"] = df_r["Data"].dt.tz_localize(None)
df_r["Giorno"] = df_r["Data"].dt.date

print(f"Commenti referendum POS/NEG: {len(df_r):,}")

def aggregato_grezzo(df):
    """Conta tutti i commenti — ogni utente pesa per quanto commenta."""
    pos = (df["STANCE"] == "POS").sum()
    neg = (df["STANCE"] == "NEG").sum()
    si  = pos / (pos + neg) * 100
    return si, 100 - si, pos, neg

def aggregato_dedup(df):
    """Una testa, un voto: un autore = un voto totale (ultimo commento del periodo)."""
    df_d = df.sort_values("Data").drop_duplicates(
        subset=["Autore"], keep="last"
    )
    pos = (df_d["STANCE"] == "POS").sum()
    neg = (df_d["STANCE"] == "NEG").sum()
    si  = pos / (pos + neg) * 100
    return si, 100 - si, pos, neg, df_d

# A) Su tutti i dati raccolti
si_g, no_g, p_g, n_g = aggregato_grezzo(df_r)
si_d, no_d, p_d, n_d, df_dedup = aggregato_dedup(df_r)

print("\n" + "=" * 65)
print("AGGREGATI — TUTTO IL PERIODO RACCOLTA")
print("=" * 65)
print(f"  GREZZO       Sì {si_g:5.1f}%  No {no_g:5.1f}%  | scarto {si_g - SI_REAL:+5.1f}pp")
print(f"  DEDUPLICATO  Sì {si_d:5.1f}%  No {no_d:5.1f}%  | scarto {si_d - SI_REAL:+5.1f}pp")
print(f"  RISULTATO REALE   Sì {SI_REAL}%   No {NO_REAL}%")

# B) Finestra di nowcasting: 14 giorni pre-voto (7-21 marzo 2026)
data_voto = pd.Timestamp("2026-03-22")
finestra_start = data_voto - pd.Timedelta(days=14)
finestra_end   = data_voto - pd.Timedelta(days=1)

df_win = df_r[(df_r["Data"] >= finestra_start) & (df_r["Data"] <= finestra_end)].copy()
si_g14, no_g14, _, _ = aggregato_grezzo(df_win)
si_d14, no_d14, _, _, _ = aggregato_dedup(df_win)

print("\n" + "=" * 65)
print(f"AGGREGATI — FINESTRA 14gg ({finestra_start:%d/%m} → {finestra_end:%d/%m})")
print("=" * 65)
print(f"  GREZZO       Sì {si_g14:5.1f}%  No {no_g14:5.1f}%  | scarto {si_g14 - SI_REAL:+5.1f}pp")
print(f"  DEDUPLICATO  Sì {si_d14:5.1f}%  No {no_d14:5.1f}%  | scarto {si_d14 - SI_REAL:+5.1f}pp")

print("\n" + "=" * 65)
print("VERIFICA H1: errore deduplicato < 2 pp ?")
print("=" * 65)
errore = abs(si_d - SI_REAL)
errore14 = abs(si_d14 - SI_REAL)
print(f"  Errore deduplicato (totale):  {errore:.1f} pp  →  {'✅ H1 confermata' if errore < 2 else '❌ sopra soglia'}")
print(f"  Errore deduplicato (14 gg):   {errore14:.1f} pp  →  {'✅ H1 confermata' if errore14 < 2 else '❌ sopra soglia'}")

# Risparmio dataset deduplicato per le sezioni successive
df_dedup.to_csv("/kaggle/working/referendum_deduplicato.csv",
                sep=";", index=False, encoding="utf-8-sig")
print(f"\n💾 Salvato referendum_deduplicato.csv — {len(df_dedup):,} righe")

## §3 — Anatomia del bias: chi sono i "rumorosi"?

Lo scarto fra aggregato grezzo e aggregato deduplicato (~2,1 pp) è la *firma quantitativa
del bias della minoranza rumorosa*. Per leggerla concretamente:

1. Distribuzione del numero di commenti per autore — ci aspettiamo una coda lunga molto pesante.
2. Stance dei top-100 commentatori — ci aspettiamo che siano disproporzionatamente collocati
   sul fronte Sì (il fronte numericamente perdente ma strutturalmente più attivo).

Se entrambe le previsioni si verificano, lo scarto +2,9 pp dell'aggregato grezzo è interamente
spiegato come over-rappresentazione della minoranza rumorosa pro-riforma.

In [ ]:
import matplotlib.pyplot as plt

# 1) Distribuzione commenti per autore (coda lunga)
commenti_per_autore = df_r.groupby("Autore").size()
print(f"Autori unici: {commenti_per_autore.shape[0]:,}")
print(f"Mediana commenti/autore: {commenti_per_autore.median():.0f}")
print(f"Media commenti/autore:   {commenti_per_autore.mean():.1f}")
print(f"Percentile 95:           {commenti_per_autore.quantile(0.95):.0f}")
print(f"Percentile 99:           {commenti_per_autore.quantile(0.99):.0f}")
print(f"Massimo:                 {commenti_per_autore.max():.0f}")

# Quanti commenti producono i top 1% degli autori?
soglia_top1pct = commenti_per_autore.quantile(0.99)
top1pct_autori = commenti_per_autore[commenti_per_autore >= soglia_top1pct]
quota_top1pct = top1pct_autori.sum() / len(df_r) * 100
print(f"\nTop 1% autori ({len(top1pct_autori):,}): producono il {quota_top1pct:.1f}% di tutti i commenti")

# 2) Stance dei top 100 commentatori
top100_autori = commenti_per_autore.nlargest(100).index
df_top100 = df_r[df_r["Autore"].isin(top100_autori)]
pos_t100 = (df_top100["STANCE"] == "POS").sum()
neg_t100 = (df_top100["STANCE"] == "NEG").sum()
si_t100 = pos_t100 / (pos_t100 + neg_t100) * 100

print(f"\n📍 Top-100 commentatori: {len(df_top100):,} commenti totali")
print(f"   Stance dei loro commenti:  Sì {si_t100:.1f}%  vs  No {100-si_t100:.1f}%")
print(f"   Confronto con aggregato deduplicato globale: Sì {si_d:.1f}%")
print(f"   → Bias dei rumorosi vs popolazione: {si_t100 - si_d:+.1f}pp")

# Plot: istogramma log-scale per visualizzare la coda
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(commenti_per_autore.values, bins=60, color="#3b82f6", alpha=0.85)
axes[0].set_yscale("log")
axes[0].set_xlabel("Commenti per autore")
axes[0].set_ylabel("Numero di autori (log)")
axes[0].set_title("Distribuzione commenti per autore — coda lunga")
axes[0].grid(alpha=0.3)

# Bar chart confronto Sì% top100 vs popolazione
labels = ["Top-100 commentatori", "Aggregato deduplicato", "Risultato reale"]
valori = [si_t100, si_d, SI_REAL]
colors = ["#ef4444", "#3b82f6", "#10b981"]
axes[1].bar(labels, valori, color=colors, alpha=0.85)
for i, v in enumerate(valori):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold")
axes[1].axhline(50, color="gray", linestyle=":", linewidth=1)
axes[1].set_ylabel("% pro-Sì")
axes[1].set_ylim(0, max(valori) * 1.15)
axes[1].set_title("Sì% top-100 vs popolazione vs voto reale")
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("/kaggle/working/anatomia_bias.png", dpi=150, bbox_inches="tight")
plt.show()

## §4 — Bilanciamento del dataset: top 10 canali YouTube e top 10 testate

Richiesto dal TODO §1.5 della tesi: visualizzare per ogni canale (e ogni testata) il volume
di commenti/articoli e la *polarizzazione* media (% Sì sui commenti POS+NEG).

Questa lettura aiuta a capire se l'aggregato YouTube è dominato da pochi canali con linea
editoriale molto schierata, o se è strutturalmente bilanciato. Per la stampa la stessa lettura
quantifica empiricamente l'allineamento editoriale del sistema mediatico italiano (Hallin & Mancini).

In [ ]:
def polarizzazione(df, group_col, top=10, label="canale"):
    """Per ogni gruppo (canale o testata): conteggio commenti, % POS, % NEG.
    Restituisce solo i top-N per volume."""
    g = df.groupby(group_col)["STANCE"].agg([
        ("totale", "size"),
        ("pos",    lambda x: (x == "POS").sum()),
        ("neg",    lambda x: (x == "NEG").sum()),
    ])
    # filtra solo gruppi con almeno 1 commento POS o NEG
    g = g[(g["pos"] + g["neg"]) > 0].copy()
    g["si_pct"] = g["pos"] / (g["pos"] + g["neg"]) * 100
    g["no_pct"] = 100 - g["si_pct"]
    return g.nlargest(top, "totale")


# A) Top 10 canali YouTube referendum
print("=" * 75)
print("📺 TOP 10 CANALI — Referendum YouTube")
print("=" * 75)
top_canali = polarizzazione(df_r, "Canale", top=10, label="canale")
for canale, row in top_canali.iterrows():
    bar = "█" * int(row["si_pct"] / 5)
    print(f"  {str(canale)[:35]:<35}  {int(row['totale']):>6,}  Sì {row['si_pct']:5.1f}%  {bar}")

# B) Top 10 testate stampa
print("\n" + "=" * 75)
print("📰 TOP 10 TESTATE — Rassegna stampa Camera")
print("=" * 75)
testata_col = next((c for c in df_sta.columns if "testata" in c.lower()), None)
df_st_pn = df_sta[df_sta["STANCE"].isin(["POS", "NEG"])].copy()
top_testate = polarizzazione(df_st_pn, testata_col, top=10, label="testata")
for testata, row in top_testate.iterrows():
    bar = "█" * int(row["si_pct"] / 5)
    print(f"  {str(testata)[:35]:<35}  {int(row['totale']):>6,}  Sì {row['si_pct']:5.1f}%  {bar}")

# Plot a barre orizzontale: confronto polarizzazione canali vs testate
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Canali
top_canali_sorted = top_canali.sort_values("si_pct")
colors_c = ["#10b981" if v >= 50 else "#ef4444" for v in top_canali_sorted["si_pct"]]
axes[0].barh(range(len(top_canali_sorted)),
             top_canali_sorted["si_pct"], color=colors_c, alpha=0.85)
axes[0].set_yticks(range(len(top_canali_sorted)))
axes[0].set_yticklabels([str(c)[:30] for c in top_canali_sorted.index], fontsize=9)
axes[0].axvline(50, color="black", linestyle=":", linewidth=1)
axes[0].axvline(SI_REAL, color="#3b82f6", linestyle="--", linewidth=1.5, label=f"Reale {SI_REAL}%")
axes[0].set_xlabel("% pro-Sì")
axes[0].set_title("Top 10 canali YouTube — polarizzazione")
axes[0].set_xlim(0, 100)
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3, axis="x")

# Testate
top_testate_sorted = top_testate.sort_values("si_pct")
colors_t = ["#10b981" if v >= 50 else "#ef4444" for v in top_testate_sorted["si_pct"]]
axes[1].barh(range(len(top_testate_sorted)),
             top_testate_sorted["si_pct"], color=colors_t, alpha=0.85)
axes[1].set_yticks(range(len(top_testate_sorted)))
axes[1].set_yticklabels([str(t)[:30] for t in top_testate_sorted.index], fontsize=9)
axes[1].axvline(50, color="black", linestyle=":", linewidth=1)
axes[1].axvline(SI_REAL, color="#3b82f6", linestyle="--", linewidth=1.5, label=f"Reale {SI_REAL}%")
axes[1].set_xlabel("% pro-Sì")
axes[1].set_title("Top 10 testate stampa — polarizzazione")
axes[1].set_xlim(0, 100)
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("/kaggle/working/top_canali_testate.png", dpi=150, bbox_inches="tight")
plt.show()

## §5 — Short vs Long video (TODO §1.5)

Il dataset di scraping (notebook 01) ha arricchito ogni video con la colonna `Is_Short`,
calcolata sulla durata ISO restituita dall'API YouTube (≤60 s = Short).

L'obiettivo è capire se il pubblico dei **Short** ha un profilo di stance diverso da quello
dei video lunghi (TED-style, talk show, conferenze, video divulgativi di Barbero o Travaglio).

L'ipotesi qualitativa: i Short, prodotti spesso da canali politici partigiani, dovrebbero
mostrare maggiore polarizzazione; i video lunghi dovrebbero ospitare un dibattito più
articolato e quindi una stance media più vicina al voto reale.

In [ ]:
# Verifica disponibilità della colonna
if "Is_Short" not in df_r.columns:
    print("⚠️ Colonna 'Is_Short' non presente nel dataset di inferenza.")
    print("   Recuperala dal dataset di raccolta (notebook 01) tramite merge su Video_ID,")
    print("   oppure salta questa sezione e annotala come limite operativo.")
else:
    # Normalizza Is_Short a booleano
    df_r["Is_Short_bool"] = df_r["Is_Short"].astype(str).str.upper().isin(
        ["TRUE", "1", "VERO", "SÌ", "SI"]
    )

    short = df_r[df_r["Is_Short_bool"]]
    long_ = df_r[~df_r["Is_Short_bool"]]

    def stats_block(sub, etichetta):
        pos = (sub["STANCE"] == "POS").sum()
        neg = (sub["STANCE"] == "NEG").sum()
        si  = pos / (pos + neg) * 100 if (pos + neg) else float("nan")
        autori = sub["Autore"].nunique()
        video  = sub["Video_ID"].nunique() if "Video_ID" in sub.columns else float("nan")
        return etichetta, len(sub), autori, video, si

    print("=" * 75)
    print("🎬 SHORT vs LONG VIDEO — distribuzione stance")
    print("=" * 75)
    print(f"{'Tipo':<10} {'Commenti':>10} {'Autori':>8} {'Video':>8} {'Sì %':>8}")
    print("-" * 75)
    for nome, n_com, n_aut, n_vid, si in [stats_block(short, "Short"),
                                          stats_block(long_, "Long")]:
        print(f"{nome:<10} {n_com:>10,} {n_aut:>8,} {n_vid:>8,} {si:>7.1f}%")

    # Stance per long sul deduplicato (una testa, un voto totale: rimuove l'effetto utenti prolifici)
    short_d = short.sort_values("Data").drop_duplicates(subset=["Autore"], keep="last")
    long_d  = long_.sort_values("Data").drop_duplicates(subset=["Autore"], keep="last")

    si_short_d = (short_d["STANCE"] == "POS").sum() / max(1,
                  (short_d["STANCE"].isin(["POS", "NEG"])).sum()) * 100
    si_long_d  = (long_d["STANCE"] == "POS").sum() / max(1,
                  (long_d["STANCE"].isin(["POS", "NEG"])).sum()) * 100

    print("\nDeduplicati (una testa, un voto totale):")
    print(f"  Short — Sì {si_short_d:.1f}%  |  scarto vs reale {si_short_d - SI_REAL:+.1f} pp")
    print(f"  Long  — Sì {si_long_d:.1f}%  |  scarto vs reale {si_long_d - SI_REAL:+.1f} pp")

    # Bar chart riepilogo
    fig, ax = plt.subplots(figsize=(8, 4.5))
    labels = ["Short\n(grezzo)", "Short\n(dedup.)", "Long\n(grezzo)",
              "Long\n(dedup.)", "Reale"]
    valori = [stats_block(short, "")[4], si_short_d,
              stats_block(long_, "")[4], si_long_d, SI_REAL]
    colors = ["#fbbf24", "#f59e0b", "#60a5fa", "#3b82f6", "#10b981"]
    ax.bar(labels, valori, color=colors, alpha=0.85)
    for i, v in enumerate(valori):
        ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold")
    ax.axhline(50, color="gray", linestyle=":", linewidth=1)
    ax.set_ylabel("% pro-Sì")
    ax.set_ylim(0, max(valori) * 1.15)
    ax.set_title("Stance Sì%: Short vs Long video — grezzo vs deduplicato")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig("/kaggle/working/short_vs_long.png", dpi=150, bbox_inches="tight")
    plt.show()

## §6 — Cross-tab utenti Governo × Referendum (TODO §1.5)

Domanda diretta dal TODO §1.5: *"in che percentuale chi è favorevole al governo ha votato Sì
o No al referendum, e viceversa?"*

Procedura:
1. Per ciascuno dei due dataset (governo, referendum), calcolo la **stance maggioritaria**
   di ogni utente: se ha più commenti POS che NEG, è classificato POS, altrimenti NEG.
2. Faccio inner join sui due insiemi di autori — ottengo gli utenti che hanno commentato
   in entrambi i corpora.
3. Costruisco la matrice di transizione 2×2 e la visualizzo come **diagramma di Sankey**.

L'interpretazione è il test sostantivo di Cap 6: se la coppia (pro-governo, pro-Sì) è
dominante e la coppia (pro-governo, pro-No) è marginale, l'allineamento politico-partitico
è forte; il consenso al governo trasla in modo prevedibile sulla posizione referendaria.

In [ ]:
import plotly.graph_objects as go
import os

WORKING_DIR = "/kaggle/working/"


def stance_maggioritaria(df, dataset_name):
    """Per ogni autore: POS se ha più POS che NEG, altrimenti NEG."""
    df = df.dropna(subset=["Autore"]).copy()
    df = df[df["STANCE"].isin(["POS", "NEG"])]
    cnt = df.groupby(["Autore", "STANCE"]).size().unstack(fill_value=0)
    for c in ["POS", "NEG"]:
        if c not in cnt.columns:
            cnt[c] = 0
    cnt[f"Stance_{dataset_name}"] = np.where(cnt["POS"] >= cnt["NEG"], "POS", "NEG")
    return cnt[[f"Stance_{dataset_name}"]].reset_index()


users_gov = stance_maggioritaria(df_gov, "Governo")
users_ref = stance_maggioritaria(df_r,   "Referendum")

print(f"Utenti unici Governo:    {len(users_gov):,}")
print(f"Utenti unici Referendum: {len(users_ref):,}")

overlap = pd.merge(users_gov, users_ref, on="Autore", how="inner")
print(f"Utenti in entrambi i mondi: {len(overlap):,}")

if len(overlap) >= 30:
    # Matrice di transizione
    matrix = overlap.groupby(["Stance_Governo", "Stance_Referendum"]).size().reset_index(name="N")

    # Calcolo percentuali condizionali
    print("\n" + "=" * 65)
    print("📊 MATRICE DI MIGRAZIONE — utenti tracciati in entrambi i corpora")
    print("=" * 65)
    for gov_stance in ["POS", "NEG"]:
        sub = matrix[matrix["Stance_Governo"] == gov_stance]
        tot = sub["N"].sum()
        for _, row in sub.iterrows():
            pct = row["N"] / tot * 100
            etichetta_gov = "Pro-Governo" if gov_stance == "POS" else "Contro-Governo"
            etichetta_ref = "Sì Referendum" if row["Stance_Referendum"] == "POS" else "No Referendum"
            print(f"  {etichetta_gov} → {etichetta_ref}: "
                  f"{int(row['N']):>5,}  ({pct:5.1f}% di {tot:,} {etichetta_gov.lower()})")

    # Sankey
    node_labels = ["Pro-Governo", "Contro-Governo",
                   "Sì Referendum", "No Referendum"]
    node_colors = ["#10b981", "#ef4444", "#3b82f6", "#f59e0b"]

    source, target, value, link_colors = [], [], [], []
    for _, row in matrix.iterrows():
        src = 0 if row["Stance_Governo"] == "POS" else 1
        tgt = 2 if row["Stance_Referendum"] == "POS" else 3
        col = "rgba(16, 185, 129, 0.4)" if row["Stance_Governo"] == "POS" \
              else "rgba(239, 68, 68, 0.4)"
        source.append(src); target.append(tgt)
        value.append(int(row["N"])); link_colors.append(col)

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=30, thickness=30,
                  line=dict(color="black", width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=source, target=target, value=value, color=link_colors,
                  hovertemplate="%{source.label} → %{target.label}<br>"
                                "Utenti: %{value}<extra></extra>")
    )])
    fig.update_layout(
        title_text=f"<b>Migrazione utenti: Consenso governo vs Referendum giustizia</b><br>"
                   f"<sub>{len(overlap):,} utenti tracciati in entrambi i dataset</sub>",
        font_size=14, plot_bgcolor="white", height=600,
    )
    fig.write_html(os.path.join(WORKING_DIR, "sankey_governo_referendum.html"))
    fig.show(renderer="iframe_connected")
    print(f"\n💾 Sankey salvato in {WORKING_DIR}sankey_governo_referendum.html")
else:
    print("⚠️ Pochi utenti in overlap — Sankey non significativo.")

## §7 — Topic modeling LDA (TODO §1.5)

Domanda dal TODO §1.5: *"di cosa parlano i commenti?"*

Applico una **Latent Dirichlet Allocation** (Gensim) su tre sotto-corpora:
1. Commenti referendum **NEG** — il fronte contrario alla riforma
2. Commenti referendum **POS** — il fronte favorevole
3. Titoli della stampa — agenda mediatica tradizionale

Per ciascun corpus estraggo 3 cluster tematici e mostro le 7 parole più rappresentative.
Il confronto delle parole-chiave fra i tre corpora rende qualitativamente leggibile la
differenza fra l'agenda della stampa e quella del pubblico online — differenza che torna
in chiave quantitativa nei test di Granger del Capitolo 6.

> **Pre-requisito Kaggle:** `gensim` e `nltk`. Le stopwords italiane vengono scaricate al volo.

In [ ]:
import re
import gensim
from gensim import corpora
from gensim.models import LdaModel
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)

stop_it = set(stopwords.words("italian"))
# Estendo con parole troppo generiche per il dominio politico
stop_it.update([
    "fatto", "detto", "solo", "essere", "fare", "video", "canale",
    "referendum", "giustizia", "riforma", "carriere", "magistratura",
    "voto", "votare", "sì", "no", "contro", "favore", "italia", "italiani",
    "governo", "meloni", "ministro", "ministra", "paese",
    "essere", "avere", "andare", "venire", "potere", "dovere",
])


def clean_lda(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\sàèéìòù]", " ", text)
    return [w for w in text.split() if w not in stop_it and len(w) > 3]


def topic_modeling(testi, nome, n_topics=3, n_words=7):
    """Estrae n_topics da una serie di testi, restituisce parole top per cluster."""
    docs = [clean_lda(t) for t in testi.dropna() if len(str(t)) > 15]
    docs = [d for d in docs if len(d) >= 3]   # almeno 3 parole post-clean
    if not docs:
        print(f"⚠️ {nome}: nessun documento utilizzabile")
        return

    dictionary = corpora.Dictionary(docs)
    dictionary.filter_extremes(no_below=5, no_above=0.5)
    bow = [dictionary.doc2bow(d) for d in docs]

    lda = LdaModel(corpus=bow, id2word=dictionary, num_topics=n_topics,
                   passes=15, random_state=42)

    print(f"\n{'=' * 75}")
    print(f"🔍 {nome} — {len(docs):,} documenti analizzati")
    print(f"{'=' * 75}")
    for i, t in lda.print_topics(num_words=n_words):
        words = re.findall(r'"(.*?)"', t)
        print(f"  Cluster {i + 1}: {' • '.join(words)}")


# Cor pus per topic modeling
testo_col_yt = "Testo" if "Testo" in df_r.columns else "Commento"

corpora_da_analizzare = [
    (df_r[df_r["STANCE"] == "NEG"][testo_col_yt], "REFERENDUM — Commenti NEG (pro-No)"),
    (df_r[df_r["STANCE"] == "POS"][testo_col_yt], "REFERENDUM — Commenti POS (pro-Sì)"),
]

# Aggiungo i titoli della stampa se la colonna esiste
titolo_col = next((c for c in df_sta.columns if "titolo" in c.lower()), None)
if titolo_col:
    corpora_da_analizzare.append(
        (df_sta[titolo_col], "STAMPA — Titoli articoli")
    )

for testi, nome in corpora_da_analizzare:
    topic_modeling(testi, nome, n_topics=3, n_words=7)

## Sintesi del notebook

| Ipotesi / Analisi | Atteso | Osservato (riempire dopo run) |
|---|---|---|
| H1 — Errore deduplicato < 2 pp | < 2 pp | _da inserire_ |
| Errore aggregato grezzo | ~ 2,9 pp | _da inserire_ |
| Distribuzione commenti coda lunga | Top 1% > 25% volume | _da inserire_ |
| Stance top-100 commentatori | > Sì popolazione | _da inserire_ |
| Top canali polarizzazione | range 20-80 % Sì | _da inserire_ |
| Top testate polarizzazione | bias pro-Sì 65-75% | _da inserire_ |
| Short vs Long stance | Short più polarizzati | _da inserire_ |
| Sankey Governo→Referendum | flusso pro-governo→Sì dominante | _da inserire_ |
| Topic LDA NEG | temi: indipendenza, separazione, controllo | _da inserire_ |
| Topic LDA POS | temi: efficienza, riforma, anni | _da inserire_ |

**File salvati in `/kaggle/working/`:**
- `referendum_deduplicato.csv` — base per i notebook successivi (turning points e Granger)
- `anatomia_bias.png` — istogramma e confronto Sì%
- `top_canali_testate.png` — bar chart polarizzazione
- `short_vs_long.png` — confronto stance per tipologia video
- `sankey_governo_referendum.html` — diagramma di flusso interattivo

**Prossimo notebook:** `04_serie_temporali_e_turning_points.ipynb` (alimenta Cap 5).